In [24]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)   

#response = llm.invoke("Hola mundo")

#print(response)

In [12]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})


    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute(message)
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self, message):
        completion = llm.invoke(message)
        return completion
    

agente = Agent(system="Eres un asistente útil y objetivo.")
print(agente("Cuál es la capital de Argentina?"))

content=[{'type': 'text', 'text': 'La capital de Argentina es **Buenos Aires** (oficialmente llamada Ciudad Autónoma de Buenos Aires).', 'extras': {'signature': 'EjQKMgERTTIPk7EADuIt4LhSaYYx8ZZc62eaThi9o8FrbRASm+tZNepiUAKy6kqN6A+AFCNV'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f5cb3-6d3e-7ce1-be69-572b426fda5c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'output_tokens': 20, 'total_tokens': 28, 'input_token_details': {'cache_read': 0}}


In [13]:
PROMPT_REACT = """
Funcionas en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta.
Usa "Pensamiento" para describir tu razonamiento.
Usa "Acción" para ejecutar herramientas - y luego retorna "PAUSA".
La "Observación" será el resultado de la acción ejecutada.
Acciones disponibles:

consultar_stock: devuelve la cantidad disponible de un articulo en el inventario (ej: "consultar_stock: teclado")

consultar_precio_producto: devuelve el precio unitario de un producto (ej: "consultar_precio_producto: mouse gamer")

Ejemplo:
Pregunta: ¿Cuántos monitores tenemos en el inventario?
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de monitores.
Acción: consultar_stock: monitor
PAUSA

Observación: Tenemos 75 monitores en el inventario.
Respuesta: Hay 75 monitores en el inventario.
""".strip()

In [21]:
from langchain_protocol import TypedDict


class EstadoAgente(TypedDict):
    pregunta: str
    historial: list[str]
    accion_pendiente: str
    respuesta_final: str


    def consultar_stock(item: str) -> str:
        """Simula la consulta de stock de item en el inventario."""

        item = item.lower()
        stock = {
            "monitor": 75,
            "teclado": 120,
            "mouse de gamer": 80,
            "webcam": 40,
            "headset": 60,
            "impresora": 15
        }

        if item in stock:
            return f"Tenemos {stock[item]} {item}s en stock."
        else:
            return f"Item '{item}' no encontrado en el inventario." 
        
    def consultar_precio_producto(producto: str) -> str:
        """Simula la consulta del precio unitario de un producto."""
        producto = producto.lower()
        precios = {
            "monitor": 999.90,
            "teclado": 150.00,
            "mouse de gamer": 99.50,
            "webcam": 120.00,
            "headset": 180.00,
            "impresora": 750.00
        }

        if producto in precios:
            return f"El precio de un(a) {producto} es USD {precios[producto]:.2f}."
        else:
            return f"producto '{producto}' no hallado en la lista de precios."
        
    print(consultar_stock("teclado"))
    print(consultar_precio_producto("impresora"))
    print(consultar_stock("monitor"))
    print(consultar_stock("sillas"))

Tenemos 120 teclados en stock.
El precio de un(a) impresora es USD 750.00.
Tenemos 75 monitors en stock.
Item 'sillas' no encontrado en el inventario.


In [22]:
    def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
        chat = llm.start_chat(history=[])
        chat.send_message(PROMPT_REACT)
        current_prompt = pregunta

        for i in range(max_iterations):
            response = chat.send_message(current_prompt)
            response_text = response.text.strip()

            print(f"--- Iteración {i+1} ---")
            print(f"Modelo pensó/respondió:\n{response_text}\n")

            if response_text.startswith("Respuesta:"):
                return response_text.replace("Respuesta:", "").strip()

            match = re.search(r"Acción:\s*(\w+)\s*\((.*)\)", response_text)

            if match:
                action_name = match.group(1).strip()
                action_arg = match.group(2).strip()
                
                observacion = ""
                if action_name == "consultar_almacen":
                    observacion = consultar_stock(action_arg)
                elif action_name == "consultar_precio_producto":
                    observacion = consultar_precio_producto(action_arg)
                else:
                    observacion = f"Error: Acción '{action_name}' desconocida."

                current_prompt = f"Observación: {observacion}\nRespuesta:"
                
                print(f"Executou acción: {action_name}({action_arg})")
                print(f"Observación: {observacion}\n")
            else:
                return f"Error: El agente no logró extraer una Acción o Respuesta final tras {i+1} iteraciones. La última respuesta fue: {response_text}"

        return "Error: Límite máximo de iteraciones alcanzado sin una respuesta final."

In [23]:
pregunta_1 = "Cuántos mouses de gamer están disponibles en el inventario?"
print(f"**Interacción 1: {pregunta_1}**")
respuesta_1 = run_react_agent(pregunta_1)
print(f"\n**RESPUESTA FINAL DEL AGENTE 1:** {respuesta_1}\n")

print("\n" + "="*50 + "\n")

**Interacción 1: Cuántos mouses de gamer están disponibles en el inventario?**


AttributeError: 'ChatGoogleGenerativeAI' object has no attribute 'start_chat'